In [ ]:
import copy, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
from scipy.signal import cont2discrete
from sklearn.neural_network import MLPRegressor
warnings.filterwarnings("ignore")

In [ ]:
# Plant: two coupled tanks, input u (V), output y = h2 (cm)
A, KP, C1, C2 = 10.0, 0.8, 1.0, 1.0
TS, NOISE = 2.0, 0.05

def ode(t, h, u, c2):
    h1, h2 = np.maximum(h, 0)
    return [(KP*u - C1*np.sqrt(h1))/A, (C1*np.sqrt(h1) - c2*np.sqrt(h2))/A]

def plant_step(h, u, c2=C2):
    u = float(np.clip(u, 0, 5))
    return np.maximum(solve_ivp(ode, (0, TS), h, args=(u, c2), rtol=1e-6, atol=1e-8).y[:, -1], 0)

def simulate(useq, h0=(0, 0), c2=C2, noise=NOISE, rng=None):
    rng = rng or np.random.default_rng()
    h, y = np.array(h0, float), []
    for u in useq:
        h = plant_step(h, u, c2)
        y.append(h[1])
    y = np.array(y)
    return y + rng.normal(0, noise, len(y)), y

In [ ]:
# Experiment 1: step responses at a low and a high level
n = 150
t = np.arange(n)*TS
_, lo = simulate(np.r_[np.full(10, 1.), np.full(n-10, 2.)], h0=(0.64, 0.64), noise=0)
_, hi = simulate(np.r_[np.full(10, 3.), np.full(n-10, 4.)], h0=(5.76, 5.76), noise=0)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(t, lo, label="1->2 V (low level)"); ax[0].plot(t, hi, label="3->4 V (high level)")
ax[0].set(xlabel="time (s)", ylabel="h2 (cm)"); ax[0].legend(); ax[0].grid()
ax[1].plot(t, (lo-lo[9])/(lo[-1]-lo[9]), label="low level")
ax[1].plot(t, (hi-hi[9])/(hi[-1]-hi[9]), label="high level")
ax[1].set(xlabel="time (s)", title="Normalised rise"); ax[1].legend(); ax[1].grid()
plt.tight_layout(); plt.savefig("exp1_nonlinearity.png", dpi=150); plt.show()

In [ ]:
# Experiment 2: dataset (random staircase + PRBS bursts)
def make_input(n, rng):
    u, k = np.zeros(n), 0
    while k < n:
        hold = rng.integers(50, 150)
        lvl = rng.uniform(0.5, 5)
        u[k:k+hold] = lvl
        k += hold
        if rng.random() < 0.3 and k < n:
            m = min(rng.integers(20, 40), n-k)
            u[k:k+m] = np.clip(lvl + rng.uniform(0.5, 1.5)*rng.choice([-1, 1], m), 0, 5)
            k += m
    return np.clip(u, 0, 5)

rng = np.random.default_rng(0)
N = 6000
u_raw = make_input(N, rng)
y_raw, yc_raw = simulate(u_raw, rng=rng)

fig, ax = plt.subplots(3, 1, figsize=(10, 7))
ax[0].plot(np.arange(N)*TS, u_raw); ax[0].set_ylabel("u (V)")
ax[1].plot(np.arange(N)*TS, y_raw); ax[1].set_ylabel("y (cm)"); ax[1].set_xlabel("time (s)")
ax[2].hist(u_raw, bins=30); ax[2].set_xlabel("u (V)")
for a in ax: a.grid()
plt.tight_layout(); plt.savefig("exp2_dataset.png", dpi=150); plt.show()

In [ ]:
# Models use a 10 s sampling interval (every 5th plant sample)
D = 5
TSM = TS*D
n = N//D*D
u = u_raw[:n].reshape(-1, D).mean(1)
y = y_raw[:n].reshape(-1, D)[:, -1]
yc = yc_raw[:n].reshape(-1, D)[:, -1]
N2 = len(u)
a, b = int(.64*N2), int(.80*N2)
u_tr, y_tr, u_va, y_va, u_te, y_te = u[:a], y[:a], u[a:b], y[a:b], u[b:], y[b:]
um, us, ym, ys = u_tr.mean(), u_tr.std(), y_tr.mean(), y_tr.std()
print(len(u_tr), len(u_va), len(u_te))

In [ ]:
def rmse(y, p): return float(np.sqrt(np.mean((np.asarray(y)-np.asarray(p))**2)))
def r2(y, p): return float(1 - np.sum((y-p)**2)/np.sum((y-np.mean(y))**2))

def regress(u, y):
    k = np.arange(2, len(y))
    return np.c_[y[k-1], y[k-2], u[k], u[k-1]], y[k]

class ARX:
    def fit(self, u, y):
        X, t = regress(u, y)
        self.th = np.linalg.lstsq(np.c_[X, np.ones(len(X))], t, rcond=None)[0]
        return self
    def predict(self, X):
        return np.c_[X, np.ones(len(X))] @ self.th

class NARX:
    def __init__(self, mlp): self.mlp = mlp
    def norm(self, X):
        return np.c_[(X[:, 0]-ym)/ys, (X[:, 1]-ym)/ys, (X[:, 2]-um)/us, (X[:, 3]-um)/us]
    def predict(self, X):
        return self.mlp.predict(self.norm(X))*ys + ym

def sig(x): return 1/(1+np.exp(-x))

class LSTM:
    def __init__(self, H=24, nin=2, seed=0):
        r = np.random.default_rng(seed); s = 1/np.sqrt(H); self.H = H
        self.p = dict(Wx=r.uniform(-s, s, (nin, 4*H)), Wh=r.uniform(-s, s, (H, 4*H)),
                      b=np.zeros(4*H), Wy=r.uniform(-s, s, (H, 1)), by=np.zeros(1))
        self.p["b"][H:2*H] = 1

    def forward(self, X, h=None, c=None):
        p, H = self.p, self.H
        B, T, _ = X.shape
        h = np.zeros((B, H)) if h is None else h
        c = np.zeros((B, H)) if c is None else c
        Y, cache = np.zeros((B, T)), []
        for t in range(T):
            z = X[:, t]@p["Wx"] + h@p["Wh"] + p["b"]
            i, f, g, o = sig(z[:, :H]), sig(z[:, H:2*H]), np.tanh(z[:, 2*H:3*H]), sig(z[:, 3*H:])
            c2 = f*c + i*g
            tc = np.tanh(c2)
            cache.append((X[:, t], h, c, i, f, g, o, tc))
            h, c = o*tc, c2
            Y[:, t] = (h@p["Wy"] + p["by"])[:, 0]
        return Y, cache, h, c

    def backward(self, cache, dY):
        p, H = self.p, self.H
        gr = {k: np.zeros_like(v) for k, v in p.items()}
        dh_n = np.zeros((dY.shape[0], H)); dc_n = np.zeros_like(dh_n)
        for t in reversed(range(len(cache))):
            x, hp, cp, i, f, g, o, tc = cache[t]
            dy = dY[:, t:t+1]
            gr["Wy"] += (o*tc).T@dy; gr["by"] += dy.sum(0)
            dh = dy@p["Wy"].T + dh_n
            dc = dh*o*(1-tc**2) + dc_n
            dz = np.c_[dc*g*i*(1-i), dc*cp*f*(1-f), dc*i*(1-g**2), dh*tc*o*(1-o)]
            dc_n = dc*f
            gr["Wx"] += x.T@dz; gr["Wh"] += hp.T@dz; gr["b"] += dz.sum(0)
            dh_n = dz@p["Wh"].T
        return gr

def onestep(model, u, y, k0):
    X, t = regress(u, y)
    return model.predict(X[k0-2:])

def freerun(model, u, y, k0):
    y1, y2, out = y[k0-1], y[k0-2], []
    for k in range(k0, len(u)):
        p = model.predict(np.array([[y1, y2, u[k], u[k-1]]]))[0]
        out.append(p); y2, y1 = y1, p
    return np.array(out)

def lstm_run(net, u, y, k0, free):
    un, yn = (u-um)/us, (y-ym)/ys
    h = c = None
    out, last = [], None
    for k in range(len(u)):
        yin = yn[max(k-1, 0)] if (not free or k <= k0 or last is None) else last
        yh, _, h, c = net.forward(np.array([[[un[k], yin]]]), h, c)
        last = yh[0, 0]; out.append(last)
    return np.array(out[k0:])*ys + ym

In [ ]:
# Train ARX and MLP-NARX
arx = ARX().fit(u_tr, y_tr)

def nx(u, y):
    X, t = regress(u, y)
    return NARX(None).norm(X), (t-ym)/ys

Xtr, ttr = nx(u_tr, y_tr)
Xva, tva = nx(u_va, y_va)
mlp = MLPRegressor(hidden_layer_sizes=(32, 32), activation="tanh", learning_rate_init=1e-3,
                   batch_size=32, random_state=0)
mlp_tr, mlp_va, best = [], [], np.inf
for ep in range(2500):
    mlp.partial_fit(Xtr, ttr)
    mlp_tr.append(np.mean((mlp.predict(Xtr)-ttr)**2))
    mlp_va.append(np.mean((mlp.predict(Xva)-tva)**2))
    if mlp_va[-1] < best:
        best, best_mlp, mlp_ep = mlp_va[-1], copy.deepcopy(mlp), ep
narx = NARX(best_mlp)
print("best MLP epoch:", mlp_ep+1)

In [ ]:
# Train LSTM (backpropagation through time + Adam)
T_WIN, STRIDE, BURN, BATCH, EPOCHS = 30, 2, 3, 32, 250
un_tr, yn_tr = (u_tr-um)/us, (y_tr-ym)/ys
un_va, yn_va = (u_va-um)/us, (y_va-ym)/ys
st = np.arange(1, len(un_tr)-T_WIN, STRIDE)
Xw = np.stack([np.c_[un_tr[s:s+T_WIN], yn_tr[s-1:s-1+T_WIN]] for s in st])
Yw = np.stack([yn_tr[s:s+T_WIN] for s in st])
Xv, Yv = np.c_[un_va[1:], yn_va[:-1]][None], yn_va[1:][None]

net = LSTM()
m_ = {k: np.zeros_like(v) for k, v in net.p.items()}
v_ = {k: np.zeros_like(v) for k, v in net.p.items()}
step, lr, r = 0, 1e-2, np.random.default_rng(1)
lstm_tr, lstm_va, best = [], [], np.inf
for ep in range(EPOCHS):
    order, losses = r.permutation(len(Xw)), []
    for i in range(0, len(order), BATCH):
        idx = order[i:i+BATCH]
        Yh, cache, _, _ = net.forward(Xw[idx])
        err = (Yh - Yw[idx])[:, BURN:]
        losses.append(np.mean(err**2))
        dY = np.zeros_like(Yh); dY[:, BURN:] = 2*err/err.size
        gr = net.backward(cache, dY)
        gn = np.sqrt(sum((g**2).sum() for g in gr.values()))
        if gn > 1: gr = {k: g/gn for k, g in gr.items()}
        step += 1
        for k in net.p:
            m_[k] = .9*m_[k] + .1*gr[k]
            v_[k] = .999*v_[k] + .001*gr[k]**2
            net.p[k] -= lr*(m_[k]/(1-.9**step))/(np.sqrt(v_[k]/(1-.999**step)) + 1e-8)
    lr = max(lr*.98, 5e-4)
    lstm_tr.append(np.mean(losses))
    lstm_va.append(np.mean((net.forward(Xv)[0] - Yv)[:, BURN:]**2))
    if lstm_va[-1] < best:
        best, best_p, lstm_ep = lstm_va[-1], {k: v.copy() for k, v in net.p.items()}, ep
net.p = best_p
print("best LSTM epoch:", lstm_ep+1)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a_, tr, va, nm in [(ax[0], mlp_tr, mlp_va, "MLP-NARX"), (ax[1], lstm_tr, lstm_va, "LSTM")]:
    a_.semilogy(tr, label="train"); a_.semilogy(va, label="validation")
    a_.set(title=nm, xlabel="epoch", ylabel="MSE (normalised)"); a_.legend(); a_.grid()
plt.tight_layout(); plt.savefig("exp3_loss_curves.png", dpi=150); plt.show()

In [ ]:
# Experiment 4: validation on the test set
K0 = 6
ref, ref_c = y_te[K0:], yc[b:][K0:]
names = ["Linear ARX", "MLP-NARX", "LSTM"]
one = {"Linear ARX": onestep(arx, u_te, y_te, K0), "MLP-NARX": onestep(narx, u_te, y_te, K0),
       "LSTM": lstm_run(net, u_te, y_te, K0, False)}
free = {"Linear ARX": freerun(arx, u_te, y_te, K0), "MLP-NARX": freerun(narx, u_te, y_te, K0),
        "LSTM": lstm_run(net, u_te, y_te, K0, True)}
print(f"{'Model':12s} {'1-step RMSE':>12s} {'1-step R2':>10s} {'free RMSE':>10s} {'free R2':>9s}")
for nm in names:
    print(f"{nm:12s} {rmse(ref, one[nm]):12.4f} {r2(ref, one[nm]):10.4f} "
          f"{rmse(ref, free[nm]):10.4f} {r2(ref, free[nm]):9.4f}")

print("\nFree-run RMSE (cm) by tank-2 level")
for lo_, hi_ in [(0, 3), (3, 6), (6, 9), (9, 12)]:
    m = (ref_c >= lo_) & (ref_c < hi_)
    print(f"{lo_:2d}-{hi_:<2d} cm  " + "  ".join(f"{nm}: {rmse(ref[m], free[nm][m]):.3f}" for nm in names))

tt = np.arange(K0, len(u_te))*TSM
fig, ax = plt.subplots(2, 1, figsize=(10, 6.5))
for a_, sl in [(ax[0], slice(None)), (ax[1], slice(0, 140))]:
    a_.plot(tt[sl], ref[sl], "k", lw=1, label="measured")
    for nm, col in zip(names, ["tab:red", "tab:blue", "tab:green"]):
        a_.plot(tt[sl], free[nm][sl], c=col, label=nm)
    a_.set_ylabel("h2 (cm)"); a_.grid()
ax[0].legend(ncol=2); ax[1].set_xlabel("time (s)")
plt.tight_layout(); plt.savefig("exp4_freerun.png", dpi=150); plt.show()

In [ ]:
# Experiment 5: PID tuned on the plant linearised at 9 cm
H0 = 9.0
a1, a2 = C1/(2*np.sqrt(H0)), C2/(2*np.sqrt(H0))
Al = np.array([[-a1/A, 0], [a1/A, -a2/A]])
Bl, Cl = np.array([[KP/A], [0]]), np.array([[0, 1.]])
Ad, Bd = cont2discrete((Al, Bl, Cl, [[0]]), TSM, "zoh")[:2]

def lin_cost(p, r=2.0, n=60):
    kp, ki, kd = np.exp(p)
    x, I, y1, y2, df, ys_, us_ = np.zeros((2, 1)), 0., 0., 0., 0., [], []
    for k in range(n):
        e = r - y1
        df = .7*df + .3*(-(y1 - y2)/TSM)
        I += ki*TSM*e
        uu = kp*e + I + kd*df
        x = Ad@x + Bd*uu
        y2, y1 = y1, float((Cl@x)[0, 0])
        ys_.append(y1); us_.append(uu)
    ys_, us_ = np.array(ys_), np.array(us_)
    if not np.all(np.isfinite(ys_)) or np.abs(ys_).max() > 1e3: return 1e9
    os_ = max(0, ys_.max()/r - 1)
    return np.sum(np.abs(r - ys_))*TSM/r + 400*max(0, os_ - .10) + 5*np.sum(np.diff(us_)**2)

sol = minimize(lin_cost, np.log([.3, .004, 5.]), method="Nelder-Mead", options=dict(maxiter=600))
KP_PID, KI_PID, KD_PID = np.exp(sol.x)
print(f"PID gains: Kp={KP_PID:.3f} V/cm  Ki={KI_PID:.5f} V/(cm s)  Kd={KD_PID:.2f} V s/cm")

In [ ]:
class PID:
    def __init__(self, kp, ki, kd): self.kp, self.ki, self.kd = kp, ki, kd
    def reset(self, r0, u0): self.I, self.df = u0, 0.
    def act(self, r, y1, y2, u1):
        e = r - y1
        self.df = .7*self.df + .3*(-(y1 - y2)/TSM)
        I = self.I + self.ki*TSM*e
        u = self.kp*e + I + self.kd*self.df
        if 0 <= u <= 5: self.I = I
        return float(np.clip(u, 0, 5))

class NNMPC:
    def __init__(self, net, H=12, N=300, lam=.5, gz=.08, seed=0):
        self.net, self.H, self.N, self.lam, self.gz = net, H, N, lam, gz
        self.rng = np.random.default_rng(seed)
    def reset(self, r0, u0):
        x = np.array([[[(u0-um)/us, (r0-ym)/ys]]])
        h = c = None
        for _ in range(15): _, _, h, c = self.net.forward(x, h, c)
        self.h, self.c, self.z, self.prev, self.r_last, self.hold = h, c, 0., np.full(self.H, u0), r0, 0
    def act(self, r, y1, y2, u1):
        net, H, N, rng = self.net, self.H, self.N, self.rng
        yh, _, self.h, self.c = net.forward(np.array([[[(u1-um)/us, (y2-ym)/ys]]]), self.h, self.c)
        if r != self.r_last: self.r_last, self.hold = r, 10
        self.hold = max(self.hold - 1, 0)
        if self.hold == 0 and 0 < u1 < 5 and abs(y1 - y2) < .12: self.z = float(np.clip(self.z + self.gz*(r - y1), -4, 4))
        shifted = np.r_[self.prev[1:], self.prev[-1]]
        seg = np.repeat(np.arange(3), H//3)
        base = [np.full(H, u1), shifted]
        g = [rng.uniform(0, 5, (N//4, 3))[:, seg],
             u1 + .6*rng.standard_normal((N//3, 3))[:, seg],
             (shifted[::H//3][:3] + .25*rng.standard_normal((N - N//4 - N//3 - 2, 3)))[:, seg]]
        U = np.clip(np.vstack(base + g), 0, 5)
        n = len(U)
        h, c, yin = np.repeat(self.h, n, 0), np.repeat(self.c, n, 0), np.full(n, (y1-ym)/ys)
        Yp = np.zeros((n, H))
        for t in range(H):
            yh, _, h, c = net.forward(np.stack([(U[:, t]-um)/us, yin], -1)[:, None, :], h, c)
            yin = yh[:, 0]; Yp[:, t] = yin*ys + ym
        dU = np.diff(np.c_[np.full(n, u1), U], axis=1)
        J = ((r + self.z - Yp)**2).sum(1) + self.lam*(dU**2).sum(1)
        self.prev = U[J.argmin()]
        return float(self.prev[0])

In [ ]:
# Closed-loop simulation on the TRUE plant (control period 10 s, sensor noise on)
N_PRE, N_POST = 8, 70

def closed_loop(ctrl, r0, r1, dist=None, noise=NOISE, seed=0):
    rng = np.random.default_rng(seed)
    h = np.array([r0, r0], float)
    u_prev = C2*np.sqrt(r0)/KP
    ctrl.reset(r0, u_prev)
    yh_ = [r0, r0]
    H2, U_, Y_ = [], [], []
    for k in range(N_PRE + N_POST):
        r = r0 if k < N_PRE else r1
        u = ctrl.act(r, yh_[-1], yh_[-2], u_prev)
        for j in range(D):
            c2 = C2*dist if (dist and k >= N_PRE) else C2
            h = plant_step(h, u, c2)
            H2.append(h[1])
        yh_.append(h[1] + rng.normal(0, noise))
        U_.append(u); u_prev = u
    return dict(h2=np.array(H2), u=np.array(U_), r0=r0, r1=r1)

def integ(f, t): return np.sum((f[1:] + f[:-1])/2*np.diff(t))

def step_metrics(res):
    r0, r1 = res["r0"], res["r1"]
    seg = np.r_[r0, res["h2"][N_PRE*D:]]
    t = np.arange(len(seg))*TS
    z = (seg - r0)/(r1 - r0)
    t10, t90 = t[np.argmax(z >= .1)], t[np.argmax(z >= .9)] if (z >= .9).any() else np.nan
    bad = np.where(np.abs(z - 1) > .05)[0]
    settle = t[bad[-1] + 1] if len(bad) and bad[-1] + 1 < len(t) else (0 if not len(bad) else np.nan)
    return dict(rise=t90 - t10, over=max(0, z.max() - 1)*100, settle=settle,
                iae=float(integ(np.abs(r1 - seg), t)))

def dist_metrics(res):
    seg, r = res["h2"][N_PRE*D:], res["r1"]
    t = np.arange(len(seg))*TS
    bad = np.where(np.abs(seg - r) > .2)[0]
    return dict(maxdev=np.abs(seg - r).max(), settle=t[bad[-1] + 1] if len(bad) and bad[-1] + 1 < len(t) else np.nan,
                iae=float(integ(np.abs(seg - r), t)), final=abs(seg[-30:].mean() - r))

In [ ]:
# Experiment 6-7: PID vs NN-MPC at three operating levels + disturbance
pid, mpc = PID(KP_PID, KI_PID, KD_PID), NNMPC(net)
tests = [(3, 5), (7, 9), (12, 14)]
R = {(nm, tst): closed_loop(c, *tst) for nm, c in [("PID", pid), ("NN-MPC", mpc)] for tst in tests}
Rd = {nm: closed_loop(c, 9, 9, dist=1.2) for nm, c in [("PID", pid), ("NN-MPC", mpc)]}

print(f"{'Step (cm)':10s} {'Ctrl':7s} {'Rise (s)':>9s} {'Overshoot %':>12s} {'Settling (s)':>13s} {'IAE (cm.s)':>11s}")
for tst in tests:
    for nm in ["PID", "NN-MPC"]:
        m = step_metrics(R[(nm, tst)])
        print(f"{tst[0]:>3d} -> {tst[1]:<4d} {nm:7s} {m['rise']:9.0f} {m['over']:12.1f} {m['settle']:13.0f} {m['iae']:11.1f}")
print("\nDisturbance: c2 +20% at 9 cm")
for nm in ["PID", "NN-MPC"]:
    m = dist_metrics(Rd[nm])
    print(f"{nm:7s} max dev {m['maxdev']:.2f} cm | settle (+-0.2 cm) {m['settle']:.0f} s | IAE {m['iae']:.1f} | final error {m['final']:.2f} cm")

fig, ax = plt.subplots(2, 4, figsize=(16, 6))
for j, tst in enumerate(tests):
    for nm, col in [("PID", "tab:red"), ("NN-MPC", "tab:blue")]:
        res = R[(nm, tst)]
        ax[0, j].plot(np.arange(len(res["h2"]))*TS - N_PRE*TSM, res["h2"], c=col, label=nm)
        ax[1, j].step(np.arange(len(res["u"]))*TSM - N_PRE*TSM, res["u"], c=col, where="post")
    ax[0, j].axhline(tst[1], c="k", ls="--", lw=.8); ax[0, j].set_title(f"{tst[0]} -> {tst[1]} cm")
for nm, col in [("PID", "tab:red"), ("NN-MPC", "tab:blue")]:
    res = Rd[nm]
    ax[0, 3].plot(np.arange(len(res["h2"]))*TS - N_PRE*TSM, res["h2"], c=col, label=nm)
    ax[1, 3].step(np.arange(len(res["u"]))*TSM - N_PRE*TSM, res["u"], c=col, where="post")
ax[0, 3].axhline(9, c="k", ls="--", lw=.8); ax[0, 3].set_title("disturbance: c2 +20% at t=0")
for a_ in ax.ravel(): a_.grid()
ax[0, 0].set_ylabel("h2 (cm)"); ax[1, 0].set_ylabel("u (V)"); ax[0, 0].legend()
for a_ in ax[1]: a_.set_xlabel("time (s)")
plt.tight_layout(); plt.savefig("exp7_control.png", dpi=150); plt.show()